# 🏋️ Fine-tuning USB AI System
## QLoRA Fine-tuning: Qwen2.5-1.5B-Instruct

> **Tugas Akhir** — Informatika, Universitas Andalas 2026  
> **Framework:** Unsloth + TRL (HuggingFace)  
> **Target:** Model GGUF portabel untuk USB AI System  

### Alur Notebook
1. 📦 Install dependensi
2. 🤗 Load base model (Qwen2.5-1.5B-Instruct)
3. ⚙️ Konfigurasi QLoRA
4. 📊 Load & format dataset
5. 🏋️ Fine-tuning
6. 💾 Save & export ke GGUF
7. 🧪 Test model hasil fine-tuning

## 0. Cek GPU Environment

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name}')
    print(f'VRAM: {gpu.total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  Tidak ada GPU. Aktifkan Runtime GPU di Colab: Runtime → Change runtime type → T4 GPU')

## 1. 📦 Install Dependensi

In [ ]:
%%capture
# Install Unsloth (optimized fine-tuning)
!pip install unsloth
!pip install --upgrade --no-cache-dir 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

# Install tools tambahan
!pip install trl peft accelerate datasets transformers
!pip install -q bitsandbytes

## 2. 🤗 Load Base Model dengan Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# ─── Konfigurasi Model ───
MODEL_NAME     = 'Qwen/Qwen2.5-1.5B-Instruct'  # Base model
MAX_SEQ_LENGTH = 2048   # Context window
DTYPE          = None   # Auto-detect (float16 untuk T4, bfloat16 untuk A100)
LOAD_IN_4BIT   = True   # QLoRA: quantize model ke 4-bit saat loading

print(f'Loading model: {MODEL_NAME}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('✅ Model berhasil dimuat!')

## 3. ⚙️ Konfigurasi LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                        # LoRA rank — lebih besar = lebih ekspresif tapi lebih lambat
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,               # Scaling factor (biasanya 2x rank)
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Hemat VRAM
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# Hitung parameter yang di-train
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} ({100 * trainable / total:.2f}% dari total)')

## 4. 📊 Load & Format Dataset

Format data menggunakan **ChatML** yang sesuai dengan Qwen2.5.

In [ ]:
from datasets import Dataset
import json

# ─── System Prompt ───
SYSTEM_PROMPT = (
    'Kamu adalah asisten AI yang membantu dan ramah. '
    'Jawab pertanyaan dengan jelas, informatif, dan dalam Bahasa Indonesia '
    'kecuali pengguna bertanya dalam Bahasa Inggris.'
)

# ─── Fungsi format ChatML ───
def format_chatml(example):
    messages = example['messages']
    # Tambah system prompt jika belum ada
    if messages[0]['role'] != 'system':
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + messages

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}


# ─── Contoh Dataset (ganti dengan dataset asli) ───
SAMPLE_DATA = [
    {
        'messages': [
            {'role': 'user', 'content': 'Apa itu machine learning?'},
            {'role': 'assistant', 'content': 'Machine learning (pembelajaran mesin) adalah cabang dari kecerdasan buatan (AI) yang memungkinkan sistem komputer untuk belajar dan meningkatkan kemampuannya secara otomatis dari pengalaman, tanpa perlu diprogram secara eksplisit untuk setiap tugas.\n\nKonsep utama machine learning:\n- **Data**: Model belajar dari kumpulan data\n- **Algoritma**: Metode matematis untuk menemukan pola\n- **Model**: Hasil pembelajaran yang dapat membuat prediksi\n- **Evaluasi**: Mengukur seberapa baik model bekerja'}
        ]
    },
    {
        'messages': [
            {'role': 'user', 'content': 'Jelaskan perbedaan AI, machine learning, dan deep learning!'},
            {'role': 'assistant', 'content': 'Ketiga istilah ini saling berkaitan dan membentuk hierarki:\n\n**🔵 Artificial Intelligence (AI)**\nKonsep paling luas — simulasi kecerdasan manusia dalam mesin. Mencakup semua teknik untuk membuat mesin "berpikir" atau bertindak cerdas.\n\n**🟢 Machine Learning (ML)**\nSubset dari AI — mesin belajar dari data tanpa diprogram secara eksplisit. Contoh: spam filter, rekomendasi film.\n\n**🔴 Deep Learning (DL)**\nSubset dari ML — menggunakan neural network berlapis-lapis (deep neural networks). Sangat efektif untuk gambar, suara, dan teks.\n\nAnalogi: AI = kecerdasan, ML = cara belajar, DL = salah satu teknik belajar yang sangat powerful.'}
        ]
    },
    {
        'messages': [
            {'role': 'user', 'content': 'Bagaimana cara kerja neural network?'},
            {'role': 'assistant', 'content': 'Neural network terinspirasi dari cara kerja otak manusia. Berikut cara kerjanya:\n\n**Komponen Utama:**\n1. **Input Layer** — menerima data mentah (piksel gambar, kata-kata teks, dll)\n2. **Hidden Layers** — lapisan tersembunyi yang memproses dan mengekstrak fitur\n3. **Output Layer** — menghasilkan prediksi/jawaban akhir\n\n**Proses:**\n1. Data masuk ke input layer\n2. Setiap neuron menghitung weighted sum dari inputnya\n3. Fungsi aktivasi menentukan apakah neuron aktif atau tidak\n4. Informasi mengalir maju (forward propagation)\n5. Error dihitung dan model diperbarui (backpropagation)\n6. Proses berulang ribuan kali hingga model akurat\n\nSederhananya: data masuk → diproses berlapis-lapis → prediksi keluar.'}
        ]
    },
]

# ─── Muat dataset ───
# OPSI A: Gunakan sample data di atas (untuk testing)
raw_dataset = Dataset.from_list(SAMPLE_DATA)

# OPSI B: Load dari file JSONL (ganti path sesuai dataset kamu)
# raw_dataset = Dataset.from_json('dataset/processed/train.jsonl')

# OPSI C: Load dari HuggingFace Hub
# from datasets import load_dataset
# raw_dataset = load_dataset('nama/dataset', split='train')

# Format dataset
formatted_dataset = raw_dataset.map(format_chatml, batched=False)
print(f'✅ Dataset siap: {len(formatted_dataset)} sampel')
print('\nContoh formatted output:')
print(formatted_dataset[0]['text'][:500], '...')

## 5. 🏋️ Fine-tuning dengan SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR = './usb-ai-finetuned'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        # ─── Training hyperparameters ───
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # Effective batch = 2 × 4 = 8
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,

        # ─── Precision ───
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),

        # ─── Logging & saving ───
        logging_steps=10,
        save_strategy='epoch',
        output_dir=OUTPUT_DIR,
        report_to='none',       # Ganti ke 'wandb' jika ingin tracking

        # ─── Optimasi memori ───
        optim='adamw_8bit',
        weight_decay=0.01,
        seed=42,
    ),
)

# Hitung estimasi waktu training
steps = (len(formatted_dataset) // trainer.args.per_device_train_batch_size) * trainer.args.num_train_epochs
print(f'Total steps estimasi: {steps}')
print(f'Mulai training...')

trainer_stats = trainer.train()
print(f'✅ Training selesai!')
print(f'Training loss akhir: {trainer_stats.training_loss:.4f}')

## 6. 🧪 Test Model Hasil Fine-tuning

In [ ]:
from unsloth.chat_templates import get_chat_template

# Aktifkan mode inferensi (lebih cepat)
FastLanguageModel.for_inference(model)

def test_model(question: str, max_tokens: int = 256):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {question}')
    print(f'A: {response}')
    print('-' * 60)

# Test dengan beberapa pertanyaan
test_model('Apa itu machine learning?')
test_model('Siapa kamu?')
test_model('What is deep learning?')

## 7. 💾 Simpan Model & Export ke GGUF

Export model ke format GGUF Q4_K_M untuk digunakan di USB.

In [ ]:
SAVE_DIR   = './usb-ai-model-merged'   # Model full (merged LoRA)
GGUF_DIR   = './usb-ai-gguf'           # Output GGUF

# ─── 7a. Simpan LoRA adapter saja (kecil, ~100MB) ───
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ LoRA adapter disimpan ke: {OUTPUT_DIR}')

# ─── 7b. Merge LoRA + Base Model & Export ke GGUF ───
print('\nExport ke GGUF (Q4_K_M)...')
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method='q4_k_m',   # Rekomendasi: balance antara kualitas & ukuran
)
print(f'✅ Model GGUF tersimpan ke: {GGUF_DIR}')

# List file yang dihasilkan
import os
for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1024**2
    print(f'  📄 {f} ({size_mb:.1f} MB)')

## 8. 📤 Upload ke HuggingFace Hub (Opsional)

In [ ]:
# OPSIONAL: Upload model ke HuggingFace Hub
# Ganti 'username' dengan username HF kamu

# from huggingface_hub import login
# login()  # Masukkan token HF kamu

# REPO_NAME = 'username/usb-ai-v1-gguf'
# model.push_to_hub_gguf(
#     REPO_NAME,
#     tokenizer,
#     quantization_method='q4_k_m',
#     token=True,
# )
# print(f'✅ Model uploaded ke: https://huggingface.co/{REPO_NAME}')

print('Upload ke HuggingFace dinonaktifkan. Uncomment cell di atas jika diperlukan.')

## 9. ⬇️ Download Model GGUF ke Lokal

Setelah selesai, download file `.gguf` dan taruh di folder `models/` di USB.

In [ ]:
import hashlib
import os

# Hitung SHA256 untuk verifikasi integritas
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith('.gguf')]

for fname in gguf_files:
    fpath = os.path.join(GGUF_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1024**2

    sha256 = hashlib.sha256()
    with open(fpath, 'rb') as f:
        while chunk := f.read(8 * 1024 * 1024):
            sha256.update(chunk)

    checksum = sha256.hexdigest()
    print(f'File: {fname}')
    print(f'Size: {size_mb:.1f} MB')
    print(f'SHA256: {checksum}')
    print()

print('📋 Salin SHA256 di atas ke models/metadata.json di USB!')
print()

# Download dari Colab ke PC
from google.colab import files
for fname in gguf_files:
    print(f'Memulai download: {fname}...')
    files.download(os.path.join(GGUF_DIR, fname))

---
## ✅ Checklist Setelah Fine-tuning

- [ ] Model GGUF berhasil dibuat
- [ ] SHA256 checksum dicatat
- [ ] File `.gguf` di-copy ke `USB_AI/models/model.gguf`
- [ ] `metadata.json` diperbarui dengan SHA256 dan info model
- [ ] Test jalankan `launcher.bat` / `launcher.sh`
- [ ] Verifikasi model dapat menjawab pertanyaan dengan benar

---
*Fine-tuning notebook untuk USB AI System — Tugas Akhir, Universitas Andalas 2026*